# In Class Lab 1: Build, Train, Deploy, Evaluate, and Explain an XGBoost Model with Amazon SageMaker

This notebook follows the SageMaker lab flow using the Adult Census Income dataset.

Objectives:
- prepare the raw Adult dataset for SageMaker XGBoost
- upload training data to Amazon S3
- train a built-in XGBoost model in SageMaker
- deploy the model to a real-time endpoint
- evaluate predictions on a held-out test set
- optionally add SHAP-based explainability for stronger reporting


## Before You Run

1. In AWS, switch to the exact region required by your lab before creating resources.
2. Create a SageMaker notebook instance using `ml.t2.medium` or `ml.t3.medium` if `t2` is unavailable.
3. Use platform identifier `notebook-al2023-v1`.
4. Open the notebook with the `conda_python3` kernel.
5. Recommended: upload unpacked files into this same folder or into an `adult/` subfolder beside the notebook:
   - `adult.data`
   - `adult.test`
   - `sagemaker_adult_income_lab.py`
6. Optional: `adult.zip` also works because the helper script can extract it automatically.
7. This notebook does not need SHAP for training. SHAP is only used at the end if you want an explainability section.


## Step 1: Install the SageMaker SDK

This cell installs or updates the SageMaker Python SDK used to create the session, upload data, train the model, deploy the endpoint, and interact with SageMaker resources.


In [ ]:
!pip install -qU sagemaker


## Step 2: Import Libraries and Helper Functions

This cell imports the packages required for the notebook and reuses the helper functions from `sagemaker_adult_income_lab.py`. Those helpers handle preprocessing, S3 upload, training setup, deployment support, and evaluation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sagemaker
from sagemaker.serializers import CSVSerializer
from sagemaker.session import TrainingInput

from sagemaker_adult_income_lab import (
    S3_PREFIX,
    WORK_DIR,
    build_estimator,
    build_features,
    evaluate_predictions,
    label_first_frame,
    load_adult_dataframe,
    predict_batches,
    resolve_dataset_paths,
    resolve_execution_role,
    save_exploration_plots,
    split_dataset,
    upload_to_s3,
    write_local_artifacts,
)

print("SageMaker SDK version:", sagemaker.__version__)
print("Notebook working directory:", Path.cwd())
print("Artifact directory:", WORK_DIR)


## Step 3: Load, Clean, Encode, and Split the Dataset

This cell locates the Adult dataset files, removes rows that contain unknown values, converts categorical features into numeric columns, creates train, validation, and test splits, and writes SageMaker-ready CSV files locally.


In [ ]:
train_path, test_path = resolve_dataset_paths()
print("Train file:", train_path)
print("Test file:", test_path)

clean_df = load_adult_dataframe(train_path, test_path)
feature_matrix, labels, display_features = build_features(clean_df)
save_exploration_plots(display_features, WORK_DIR)

splits = split_dataset(feature_matrix, labels)
local_artifacts = write_local_artifacts(splits, WORK_DIR)
local_artifacts


## Step 4: Review Dataset Statistics

This cell shows descriptive statistics for the cleaned feature set. It helps justify the preprocessing stage and confirms the data was loaded correctly before training begins.


In [ ]:
display_features.describe()


## Step 5: Preview the Training Format

SageMaker built-in XGBoost expects the target label in the first column and numeric feature values after it. This cell previews the transformed training dataframe in that format.


In [ ]:
train_preview = label_first_frame(splits["y_train"], splits["x_train"])
validation_preview = label_first_frame(splits["y_val"], splits["x_val"])
test_preview = label_first_frame(splits["y_test"], splits["x_test"])

train_preview.head()


## Step 6: Start the SageMaker Session

This cell discovers the AWS region, execution role, and default S3 bucket from the notebook environment. These values are required for training and deployment.


In [ ]:
session = sagemaker.Session()
region = session.boto_region_name
role = resolve_execution_role()
bucket = session.default_bucket()

print("AWS Region:", region)
print("Role ARN:", role)
print("Default bucket:", bucket)


## Step 7: Upload the Prepared CSV Files to Amazon S3

This cell uploads the generated train, validation, and test CSV files to your SageMaker default bucket. After this runs, you can open the Amazon S3 console and verify the files under the printed bucket and prefix.


In [ ]:
s3_artifacts = upload_to_s3(bucket, S3_PREFIX, local_artifacts)
s3_artifacts


## Step 8: Train the Built-In SageMaker XGBoost Model

This cell configures the built-in XGBoost container, points it at the training and validation CSV files in S3, and launches the SageMaker training job. Wait for the job to finish before continuing.


In [ ]:
xgb_model = build_estimator(session, role, region, bucket, S3_PREFIX)
train_input = TrainingInput(s3_artifacts["train"], content_type="csv")
validation_input = TrainingInput(s3_artifacts["validation"], content_type="csv")

xgb_model.fit({"train": train_input, "validation": validation_input}, wait=True)
print("Training job name:", xgb_model.latest_training_job.job_name)
print("Model artifact:", xgb_model.model_data)


## Step 9: Deploy the Trained Model

This cell deploys the trained model to a real-time SageMaker endpoint. The endpoint is billable while it is running, so remember to delete it during cleanup.


In [ ]:
xgb_predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    serializer=CSVSerializer(),
)
print("Endpoint name:", xgb_predictor.endpoint_name)


## Step 10: Evaluate the Deployed Model

This cell sends the test set to the hosted endpoint, collects prediction probabilities, computes classification metrics, and saves evaluation plots locally.


In [ ]:
predictions = predict_batches(xgb_predictor, splits["x_test"])
metrics_summary = evaluate_predictions(predictions, splits["y_test"], WORK_DIR)
metrics_summary


## Step 11: Collect Submission Values

This cell gathers the main values you will likely need in your report or submission, including the bucket, prefix, training job, endpoint name, and core evaluation metrics.


In [ ]:
submission_summary = {
    "s3_bucket": bucket,
    "s3_prefix": S3_PREFIX,
    "training_job_name": xgb_model.latest_training_job.job_name,
    "endpoint_name": xgb_predictor.endpoint_name,
    "accuracy_at_0_5": metrics_summary["accuracy_at_0_5"],
    "roc_auc": metrics_summary["roc_auc"],
    "best_cutoff": metrics_summary["best_cutoff"],
}
submission_summary


## Submission Summary

Fill this in before you submit. Paste the values from the output above.

- AWS Region:
- Notebook Instance Name:
- S3 Bucket:
- S3 Prefix:
- Training Job Name:
- Endpoint Name:
- Accuracy at cutoff 0.5:
- ROC AUC:
- Best Cutoff:
- Notes / observations:


## Optional Batch Transform

The next cell is optional. It shows how to run batch inference through SageMaker Batch Transform instead of using a hosted endpoint for every prediction request.


In [ ]:
# Uncomment if your instructor wants batch inference evidence as well.
# batch_output = f"s3://{bucket}/{S3_PREFIX}/batch-prediction"
# transformer = xgb_model.transformer(
#     instance_count=1,
#     instance_type="ml.m4.xlarge",
#     output_path=batch_output,
# )
# transformer.transform(
#     data=s3_artifacts["test_features"],
#     data_type="S3Prefix",
#     content_type="text/csv",
#     split_type="Line",
# )
# transformer.wait()
# print(batch_output)


## Optional SHAP Explainability

This section is optional but useful if you want a stronger and more professional explanation of the model's behavior.

Important note:
- the SageMaker built-in XGBoost model is trained remotely
- SHAP works most naturally with a local model object
- to keep the workflow simple, the next two cells fit a small local XGBoost model on the same processed training data and use SHAP to explain feature influence

This does not replace the SageMaker model. It adds an explainability layer to support your discussion of results.


### Install SHAP and Local XGBoost Support

This cell installs the packages needed for the explainability section. Run it only if you want to include SHAP analysis in your final notebook or report.


In [ ]:
!pip install -q shap xgboost


### Train a Local Explainer Model

This cell fits a local `xgboost.XGBClassifier` on the same encoded training data. The local model is used only for interpretation so that SHAP can inspect feature contributions directly.


In [ ]:
import xgboost as xgb

explainer_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=1,
)
explainer_model.fit(splits["x_train"], splits["y_train"])
print("Local explainer model trained.")


### Generate SHAP Values and Inspect Important Features

This cell computes SHAP values for a sample of the test set, saves a SHAP summary plot, and prints the most influential features by mean absolute SHAP value.


In [ ]:
import shap

sample = splits["x_test"].sample(min(300, len(splits["x_test"])), random_state=1)
explainer = shap.TreeExplainer(explainer_model)
shap_values = explainer.shap_values(sample)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, sample, show=False, max_display=15)
plt.tight_layout()
shap_plot_path = WORK_DIR / "shap_summary.png"
plt.savefig(shap_plot_path, bbox_inches="tight")
plt.show()
plt.close()

feature_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=sample.columns,
).sort_values(ascending=False)

print("Saved SHAP summary plot to:", shap_plot_path)
feature_importance.head(10)


## Cleanup

After you finish collecting evidence, delete the SageMaker endpoint, endpoint configuration, model, notebook instance, S3 objects under your lab prefix, and any `/aws/sagemaker/` CloudWatch log groups you no longer need.


## Screenshot Checkpoints

Capture screenshots at these points:
1. SageMaker notebook instance creation page showing region, instance type, platform identifier, and IAM role.
2. Notebook instance status `InService`.
3. Jupyter notebook open with the `conda_python3` kernel selected and your uploaded dataset files visible.
4. Dataset load and split output showing clean row counts and generated CSV paths.
5. S3 upload confirmation showing `train.csv` and `validation.csv` in your bucket.
6. Training job page with status `Completed`.
7. Endpoint page with the deployed endpoint name and `InService` status.
8. Evaluation output showing the confusion matrix, classification report, ROC AUC, and best cutoff.
9. Final `Submission Summary` section filled in.
10. Optional SHAP summary plot and top features output.
11. Cleanup confirmation showing deleted endpoint and stopped or deleted notebook instance.
